# Problem 1 - RNN, LSTM and GRU on sequences
**Student ID:** 2026AJ05899 | **h = 96** | **seeds = 7, 21, 42**

**How to run:** Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all. The notebook downloads UCI HAR itself, trains everything and saves `results.csv`, `requirements.txt` and `figures/`. A restart is safe: finished runs are skipped. The last cell bundles everything into `problem1_outputs.zip`.

Sections: Task 1 data pipelines, Task 2 cells from scratch, Task 3 HAR, Task 4 adding problem, Task 5 gradients/ablation/analysis.

In [ ]:
# Config (state these values on the report title page and in the README)
H = 96                  # hidden size for RNN, LSTM, GRU
SEEDS = [7, 21, 42]     # training seeds
D_IN = 9                # HAR channels

In [ ]:
# [setup]
import os, random, urllib.request, zipfile
import numpy as np
import torch
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run again (CPU will be far too slow).")
os.makedirs("figures", exist_ok=True)

In [ ]:
# [download HAR] relative paths only
if not os.path.isdir("har"):
    if not os.path.exists("har.zip"):
        url = ("https://archive.ics.uci.edu/static/public/240/"
               "human+activity+recognition+using+smartphones.zip")
        try:
            urllib.request.urlretrieve(url, "har.zip")
        except Exception as e:
            raise RuntimeError("HAR download failed (" + str(e) + "). Download the zip from the UCI page "
                               "(archive.ics.uci.edu/dataset/240), upload it to the Colab file panel as 'har.zip', "
                               "and run this cell again.")
    zipfile.ZipFile("har.zip").extractall("har")
    for r, _, fs in list(os.walk("har")):               # the archive may hold a second zip
        for f in fs:
            if f.endswith(".zip") and "__MACOSX" not in r:
                zipfile.ZipFile(os.path.join(r, f)).extractall("har")

# the real dataset folder: skip __MACOSX stubs, require the Inertial Signals folder
HAR_ROOT = next(r for r, d, _ in os.walk("har")
                if "__MACOSX" not in r and os.path.isdir(os.path.join(r, "train", "Inertial Signals")))
print("HAR root:", HAR_ROOT)

In [ ]:
# [load HAR]
CHANNELS = ["body_acc_x", "body_acc_y", "body_acc_z",
            "body_gyro_x", "body_gyro_y", "body_gyro_z",
            "total_acc_x", "total_acc_y", "total_acc_z"]   # fixed order
CLASSES = ["walking", "walking_upstairs", "walking_downstairs",
           "sitting", "standing", "laying"]

def load_har_split(split):
    sig = [np.loadtxt(os.path.join(HAR_ROOT, split, "Inertial Signals",
                                   f"{c}_{split}.txt")) for c in CHANNELS]
    X = np.stack(sig, axis=-1).astype(np.float32)             # (N, 128, 9)
    y = np.loadtxt(os.path.join(HAR_ROOT, split, f"y_{split}.txt")).astype(int) - 1  # 0..5
    s = np.loadtxt(os.path.join(HAR_ROOT, split, f"subject_{split}.txt")).astype(int)
    return X, y, s

X_tr_all, y_tr_all, s_tr_all = load_har_split("train")
X_te, y_te, s_te = load_har_split("test")
assert X_tr_all.shape == (7352, 128, 9) and X_te.shape == (2947, 128, 9)
assert len(np.unique(s_tr_all)) == 21 and len(np.unique(s_te)) == 9
assert set(np.unique(s_tr_all)).isdisjoint(set(np.unique(s_te)))   # no subject overlap

In [ ]:
# [subject-wise split + training-only normalisation]
def make_har_split(seed):
    """Hold out 4 of the 21 training subjects for validation (subject-wise)."""
    rng = np.random.default_rng(seed)
    train_subjects = np.unique(s_tr_all)
    val_subjects = rng.choice(train_subjects, 4, replace=False)
    val_mask = np.isin(s_tr_all, val_subjects)
    Xtr, ytr = X_tr_all[~val_mask], y_tr_all[~val_mask]
    Xva, yva = X_tr_all[val_mask], y_tr_all[val_mask]
    # per-channel mean/std from the training part ONLY
    # computed in float64: float32 accumulation over ~600k values per channel drifts
    mean = Xtr.astype(np.float64).mean(axis=(0, 1), keepdims=True)   # (1, 1, 9)
    std = Xtr.astype(np.float64).std(axis=(0, 1), keepdims=True) + 1e-8
    norm = lambda a: ((a - mean) / std).astype(np.float32)
    return dict(Xtr=norm(Xtr), ytr=ytr, Xva=norm(Xva), yva=yva,
                Xte=norm(X_te), yte=y_te, val_subjects=val_subjects,
                mean=mean, std=std)

# test: no subject leakage, stats come from train only
d = make_har_split(0)
val_mask = np.isin(s_tr_all, d["val_subjects"])
assert not set(s_tr_all[~val_mask]) & set(s_tr_all[val_mask])
m_chk = d["Xtr"].astype(np.float64).mean(axis=(0, 1))
s_chk = d["Xtr"].astype(np.float64).std(axis=(0, 1))
print("max |mean|:", np.abs(m_chk).max(), "| max |std-1|:", np.abs(s_chk - 1).max())
assert np.allclose(m_chk, 0, atol=1e-3)
assert np.allclose(s_chk, 1, atol=1e-3)
print("val subjects (seed 0):", d["val_subjects"],
      "| train/val/test:", len(d["ytr"]), len(d["yva"]), len(d["yte"]))

In [ ]:
# [class-balance plot]
fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for a, (name, labels) in zip(ax, [("train", d["ytr"]), ("val", d["yva"]), ("test", d["yte"])]):
    a.bar(range(6), np.bincount(labels, minlength=6))
    a.set_xticks(range(6)); a.set_xticklabels(CLASSES, rotation=60, ha="right", fontsize=8)
    a.set_title(f"{name} (n={len(labels)})")
ax[0].set_ylabel("windows")
plt.tight_layout(); plt.savefig("figures/har_class_balance.png", dpi=150); plt.show()

In [ ]:
# [adding problem generator - provided function, unchanged]
def make_adding(n, T, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)

In [ ]:
# [adding problem test]
def test_adding(T):
    x, y = make_adding(1000, T, seed=123)
    marker, value = x[:, :, 1], x[:, :, 0]
    assert marker.sum(axis=1).tolist() == [2.0] * len(x)       # exactly two markers
    assert (marker[:, : T // 2].sum(axis=1) == 1).all()        # one in first half
    assert (marker[:, T // 2:].sum(axis=1) == 1).all()         # one in second half
    assert np.allclose(y, (value * marker).sum(axis=1), atol=1e-6)  # target = sum of marked values

for T in (50, 100, 200, 400):
    test_adding(T)
print("adding-problem tests passed")

# splits per T: 10,000 train (seed 0), 1,000 val (seed 1), 1,000 test (seed 2)
def adding_splits(T):
    return {"train": make_adding(10_000, T, 0),
            "val":   make_adding(1_000, T, 1),
            "test":  make_adding(1_000, T, 2)}

# baseline: constant prediction of 1.0 -> MSE about 0.167
_, y_t = make_adding(1000, 200, 2)
print("constant-1.0 baseline MSE:", float(np.mean((y_t - 1.0) ** 2)))

## Settings for Tasks 2-5

In [ ]:
# [imports and experiment settings]
import math, time, json, shutil
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

HAR_EPOCHS, HAR_BS = 20, 64        # same budget for RNN / LSTM / GRU
ADD_EPOCHS, ADD_BS = 100, 128
LR, CLIP = 1e-3, 1.0               # Adam lr 1e-3; grad-norm clipping 1.0 for ALL models (stated in report)
KINDS = ["rnn", "lstm", "gru"]
NAMES = {"rnn": "RNN", "lstm": "LSTM", "gru": "GRU"}
LIB = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}

## Task 2 - cells from scratch

In [ ]:
# [Task 2: the three cells]
def _init(rows, cols, h):
    k = 1.0 / math.sqrt(h)                      # PyTorch's default: U(-1/sqrt(h), 1/sqrt(h))
    return nn.Parameter(torch.empty(rows, cols).uniform_(-k, k)) if cols else \
           nn.Parameter(torch.empty(rows).uniform_(-k, k))

class MyRNNCell(nn.Module):
    """h' = tanh(x W_ih^T + b_ih + h W_hh^T + b_hh).  x:(B,d)  h:(B,H)"""
    def __init__(self, d_in, h):
        super().__init__()
        self.h = h
        self.W_ih, self.W_hh = _init(h, d_in, h), _init(h, h, h)
        self.b_ih, self.b_hh = _init(h, 0, h), _init(h, 0, h)
    def init_state(self, B, dev): return torch.zeros(B, self.h, device=dev)
    def forward(self, x, h):
        return torch.tanh(x @ self.W_ih.T + self.b_ih + h @ self.W_hh.T + self.b_hh)

class MyLSTMCell(nn.Module):
    """Gates stacked (i, f, g, o) like PyTorch. state = (h, c), each (B,H)."""
    def __init__(self, d_in, h):
        super().__init__()
        self.h = h
        self.W_ih, self.W_hh = _init(4 * h, d_in, h), _init(4 * h, h, h)
        self.b_ih, self.b_hh = _init(4 * h, 0, h), _init(4 * h, 0, h)
    def init_state(self, B, dev):
        return (torch.zeros(B, self.h, device=dev), torch.zeros(B, self.h, device=dev))
    def forward(self, x, state):
        h, c = state
        pre = x @ self.W_ih.T + self.b_ih + h @ self.W_hh.T + self.b_hh      # (B,4H)
        i, f, g, o = pre.chunk(4, dim=1)                                      # each (B,H)
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return (h_new, c_new)

class MyGRUCell(nn.Module):
    """Gates stacked (r, z, n) like PyTorch. n = tanh(W_n x + b_in + r*(U_n h + b_hn))."""
    def __init__(self, d_in, h):
        super().__init__()
        self.h = h
        self.W_ih, self.W_hh = _init(3 * h, d_in, h), _init(3 * h, h, h)
        self.b_ih, self.b_hh = _init(3 * h, 0, h), _init(3 * h, 0, h)
    def init_state(self, B, dev): return torch.zeros(B, self.h, device=dev)
    def forward(self, x, h):
        gi = x @ self.W_ih.T + self.b_ih                                      # (B,3H)
        gh = h @ self.W_hh.T + self.b_hh                                      # (B,3H)
        i_r, i_z, i_n = gi.chunk(3, dim=1)
        h_r, h_z, h_n = gh.chunk(3, dim=1)
        r = torch.sigmoid(i_r + h_r)
        z = torch.sigmoid(i_z + h_z)
        n = torch.tanh(i_n + r * h_n)
        return (1 - z) * n + z * h

CELLS = {"rnn": MyRNNCell, "lstm": MyLSTMCell, "gru": MyGRUCell}
h_of = lambda state: state[0] if isinstance(state, tuple) else state

class MyRecurrent(nn.Module):
    """Runs a cell over a (B,T,d) sequence. Returns all hidden states (B,T,H) and the final state."""
    def __init__(self, kind, d_in, h):
        super().__init__()
        self.cell = CELLS[kind](d_in, h)
    def forward(self, x):
        state = self.cell.init_state(x.shape[0], x.device)
        outs = []
        for t in range(x.shape[1]):
            state = self.cell(x[:, t], state)
            outs.append(h_of(state))
        return torch.stack(outs, dim=1), state

### Verification against nn.RNN / nn.LSTM / nn.GRU and parameter counts
Hand derivation (write in the report): single-bias count RNN = h(d+h+1), LSTM = 4h(d+h+1), GRU = 3h(d+h+1). PyTorch keeps two bias vectors (b_ih and b_hh), adding h, 4h and 3h parameters respectively.

In [ ]:
# [Task 2: weight copy + match check (< 1e-5) + parameter counts]
def copy_weights(lib, mine):
    """Copy nn.RNN/LSTM/GRU (1 layer) weights into MyRecurrent."""
    with torch.no_grad():
        mine.cell.W_ih.copy_(lib.weight_ih_l0); mine.cell.W_hh.copy_(lib.weight_hh_l0)
        mine.cell.b_ih.copy_(lib.bias_ih_l0);   mine.cell.b_hh.copy_(lib.bias_hh_l0)

torch.manual_seed(0)
x_chk = torch.randn(8, 128, D_IN)
for kind in KINDS:
    lib = LIB[kind](D_IN, H, batch_first=True)
    mine = MyRecurrent(kind, D_IN, H)
    copy_weights(lib, mine)
    with torch.no_grad():
        out_lib, _ = lib(x_chk)
        out_mine, _ = mine(x_chk)
    err = (out_lib - out_mine).abs().max().item()
    print(f"{NAMES[kind]:5s} max |lib - mine| over all 128 steps = {err:.2e}")
    assert err < 1e-5, f"{kind} cell does not match the library layer"

# parameter counts. By hand (single bias):  RNN h(d+h+1),  LSTM 4h(d+h+1),  GRU 3h(d+h+1)
GATES = {"rnn": 1, "lstm": 4, "gru": 3}
rows_pc = []
for h_ in (64, H):
    for kind in KINDS:
        by_hand = GATES[kind] * h_ * (D_IN + h_ + 1)
        lib_n = sum(p.numel() for p in LIB[kind](D_IN, h_, batch_first=True).parameters())
        mine_n = sum(p.numel() for p in MyRecurrent(kind, D_IN, h_).parameters())
        assert mine_n == lib_n == by_hand + GATES[kind] * h_   # PyTorch keeps a 2nd bias vector
        rows_pc.append(dict(h=h_, model=NAMES[kind], by_hand_one_bias=by_hand,
                            code_count=mine_n, pytorch_count=lib_n, extra_bias=GATES[kind] * h_))
pc_table = pd.DataFrame(rows_pc); print(pc_table.to_string(index=False))
# (expected for h=64: RNN 4,736 -> 4,800; LSTM 18,944 -> 19,200; GRU 14,208 -> 14,400)

## Shared training code (Tasks 3-5)
Library `nn.RNN/LSTM/GRU` are used for the comparison, as the ground rules allow. All three models share hidden size, Adam lr 1e-3, batch size, epoch budget and gradient-norm clipping 1.0. `results.csv` is rewritten after every run and finished runs are skipped on restart.

In [ ]:
# [model, training loop, logging]
class RecModel(nn.Module):
    """Library recurrent layer (allowed for the comparison) + linear head on the LAST hidden state."""
    def __init__(self, kind, d_in, h, n_out):
        super().__init__()
        self.rec = LIB[kind](d_in, h, batch_first=True)
        self.fc = nn.Linear(h, n_out)
    def forward(self, x):
        out, _ = self.rec(x)            # (B,T,H)
        return self.fc(out[:, -1])      # last hidden state h_T

_ADD_CACHE = {}
def get_adding(T):
    if T not in _ADD_CACHE:
        _ADD_CACHE[T] = adding_splits(T)
    return _ADD_CACHE[T]

tt = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=device)

@torch.no_grad()
def predict(model, X, bs=1000):
    model.eval()
    return torch.cat([model(X[i:i + bs]) for i in range(0, len(X), bs)])

def run(kind, task, seed, h, epochs, bs, T=None, forget_bias=None, clip=CLIP, return_model=False):
    set_seed(seed)
    if task == "har":
        d = make_har_split(seed)
        Xtr, ytr = tt(d["Xtr"]), tt(d["ytr"], torch.long)
        Xva, yva = tt(d["Xva"]), tt(d["yva"], torch.long)
        Xte, yte = tt(d["Xte"]), tt(d["yte"], torch.long)
        n_out, d_in = 6, D_IN
    else:
        sp = get_adding(T)
        (Xtr, ytr), (Xva, yva), (Xte, yte) = [(tt(a), tt(b)) for a, b in (sp["train"], sp["val"], sp["test"])]
        n_out, d_in = 1, 2
    model = RecModel(kind, d_in, h, n_out).to(device)
    if forget_bias is not None and kind == "lstm":      # gates are stacked (i, f, g, o): forget = [h:2h]
        with torch.no_grad():
            model.rec.bias_ih_l0[h:2 * h].fill_(forget_bias)
            model.rec.bias_hh_l0[h:2 * h].zero_()       # so the effective forget bias is exactly forget_bias
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = (lambda o, y: F.cross_entropy(o, y)) if task == "har" else (lambda o, y: F.mse_loss(o.squeeze(1), y))
    def val_metric(m):
        o = predict(m, Xva)
        return (o.argmax(1) == yva).float().mean().item() if task == "har" else F.mse_loss(o.squeeze(1), yva).item()
    better = (lambda a, b: a > b) if task == "har" else (lambda a, b: a < b)
    best, best_state, val_curve, train_curve, sec = None, None, [], [], []
    N = len(Xtr)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(N, device=device); tot = 0.0
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for i in range(0, N, bs):
            idx = perm[i:i + bs]
            loss = loss_fn(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward()
            if clip: nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step(); tot += loss.item() * len(idx)
        if device == "cuda": torch.cuda.synchronize()
        sec.append(time.time() - t0); train_curve.append(tot / N)
        v = val_metric(model); val_curve.append(v)
        if best is None or better(v, best):
            best = v; best_state = {k: t.clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)                   # test the best-validation epoch
    out = predict(model, Xte)
    res = dict(kind=kind, task=task, seed=seed, H=h, T=T, forget_bias=forget_bias, clip=clip,
               epochs=epochs, params=sum(p.numel() for p in model.parameters()),
               sec_per_epoch=float(np.mean(sec)), best_val=best,
               val_curve=json.dumps(val_curve), train_curve=json.dumps(train_curve))
    if task == "har":
        pred = out.argmax(1).cpu().numpy(); yt = yte.cpu().numpy()
        res.update(test_acc=float((pred == yt).mean()), test_f1=float(f1_score(yt, pred, average="macro")),
                   cm=json.dumps(confusion_matrix(yt, pred, labels=list(range(6))).tolist()))
    else:
        res.update(test_mse=float(F.mse_loss(out.squeeze(1), yte).item()))
    return (res, model) if return_model else res

# results.csv is rewritten after EVERY run, and finished runs are skipped on restart
RESULTS_CSV = "results_p1.csv"
rows = pd.read_csv(RESULTS_CSV).to_dict("records") if os.path.exists(RESULTS_CSV) else []
done_ids = {r["run_id"] for r in rows}

def run_logged(exp, kind, task, seed, h, **kw):
    rid = f"{exp}|{kind}|{seed}|{h}|{kw.get('T')}|{kw.get('forget_bias')}"
    if rid in done_ids:
        return
    epochs, bs = (HAR_EPOCHS, HAR_BS) if task == "har" else (ADD_EPOCHS, ADD_BS)
    res = run(kind, task, seed, h, epochs, bs, **kw)
    res.update(exp=exp, run_id=rid)
    rows.append(res); done_ids.add(rid)
    pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
    print(rid, "->", {k: round(res[k], 4) for k in ("test_acc", "test_f1", "test_mse") if k in res},
          f"{res['sec_per_epoch']:.2f}s/ep")

## Task 3 - HAR classification

In [ ]:
# [Task 3: run]
for seed in SEEDS:
    for kind in KINDS:
        run_logged("har_main", kind, "har", seed, H)

In [ ]:
# [Task 3: table + confusion matrix of the best model]
df = pd.DataFrame(rows)
ms = lambda s, f="{:.4f}": f.format(s.mean()) + " ± " + f.format(s.std())    # std over seeds (ddof=1)
har = df[df.exp == "har_main"]
har_table = har.groupby("kind").agg(
    test_acc=("test_acc", ms), macro_F1=("test_f1", ms), params=("params", "first"),
    sec_per_epoch=("sec_per_epoch", lambda s: f"{s.mean():.2f}")).loc[KINDS]
print(har_table.to_string())

best_kind = har.groupby("kind").test_acc.mean().idxmax()
best_row = har[har.kind == best_kind].sort_values("test_acc").iloc[-1]
cm = np.array(json.loads(best_row.cm))
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(cm, cmap="Blues")
for i in range(6):
    for j in range(6):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xticks(range(6)); ax.set_xticklabels(CLASSES, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(6)); ax.set_yticklabels(CLASSES, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"{NAMES[best_kind]} (seed {int(best_row.seed)}), test acc {best_row.test_acc:.3f}")
plt.tight_layout(); plt.savefig("figures/har_confusion.png", dpi=150); plt.show()

In [ ]:
# [Adding-problem budget: drop old shorter runs so they are re-run with ADD_EPOCHS]
# (the first 40-epoch attempt stalled at the 0.167 baseline for T = 200; keep it as evidence)
if os.path.exists(RESULTS_CSV) and not os.path.exists("results_adding_40ep.csv"):
    shutil.copy(RESULTS_CSV, "results_adding_40ep.csv")
stale = lambda r: str(r["exp"]).startswith(("add_main", "abl_forget")) and r["epochs"] != ADD_EPOCHS
rows = [r for r in rows if not stale(r)]
done_ids = {r["run_id"] for r in rows}
pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
print("kept", len(rows), "finished runs; adding-problem runs will use", ADD_EPOCHS, "epochs")

## Task 4 - adding problem

In [ ]:
# [Task 4: run (T = 50, 100, 200, 400 x 3 models x 3 seeds)]
for T in (50, 100, 200, 400):
    for seed in SEEDS:
        for kind in KINDS:
            run_logged("add_main", kind, "add", seed, H, T=T)

In [ ]:
# [Task 4: table, MSE-vs-T plot, learning curves at T = 200]
df = pd.DataFrame(rows)
add = df[df.exp == "add_main"].copy()
add["T"] = add["T"].astype(int)
add["solved"] = add.test_mse < 0.05
add_table = add.groupby(["T", "kind"]).agg(
    test_mse=("test_mse", lambda s: f"{s.mean():.4f} ± {s.std():.4f}"),
    solved=("solved", lambda s: f"{int(s.sum())}/{len(s)}")).sort_index()
print(add_table.to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for kind in KINDS:
    g = add[add.kind == kind].groupby("T").test_mse
    ax[0].errorbar(g.mean().index, g.mean(), yerr=g.std(), marker="o", capsize=3, label=NAMES[kind])
ax[0].axhline(0.167, color="gray", ls="--", label="baseline 0.167"); ax[0].axhline(0.05, color="green", ls=":", label="solved < 0.05")
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("sequence length T"); ax[0].set_ylabel("test MSE"); ax[0].legend(fontsize=8)
for kind in KINDS:
    curves = np.array([json.loads(c) for c in add[(add["T"] == 200) & (add.kind == kind)].val_curve])
    m, s = curves.mean(0), curves.std(0); ep = np.arange(1, len(m) + 1)
    ax[1].plot(ep, m, label=NAMES[kind]); ax[1].fill_between(ep, m - s, m + s, alpha=0.2)
ax[1].axhline(0.167, color="gray", ls="--"); ax[1].set_yscale("log")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation MSE"); ax[1].set_title("T = 200"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/adding_mse_and_curves.png", dpi=150); plt.show()

## Task 5 - gradients, ablations, analysis

In [ ]:
# [5a: gradient norm ||dL/dh_t|| vs t at T = 100, before and after training]
def grad_profile(kind, lib_rec, fc, x, y):
    """Loss gradient w.r.t. every hidden state h_t, via my own cells (so each h_t is a graph node)."""
    mine = MyRecurrent(kind, lib_rec.input_size, lib_rec.hidden_size).to(device)
    copy_weights(lib_rec, mine)
    state = mine.cell.init_state(x.shape[0], device)
    hs = []
    for t in range(x.shape[1]):
        state = mine.cell(x[:, t], state)
        h_t = h_of(state); h_t.retain_grad(); hs.append(h_t)
    loss = F.mse_loss(fc(hs[-1]).squeeze(1), y)
    loss.backward()
    return np.array([h_t.grad.norm(dim=1).mean().item() for h_t in hs])

T_G = 100
sp = get_adding(T_G)
xg, yg = tt(sp["val"][0][:256]), tt(sp["val"][1][:256])
prof = {"before": {}, "after": {}}
for kind in KINDS:
    set_seed(SEEDS[0])
    fresh = RecModel(kind, 2, H, 1).to(device)                       # untrained
    prof["before"][kind] = grad_profile(kind, fresh.rec, fresh.fc, xg, yg)
    _, trained = run(kind, "add", SEEDS[0], H, ADD_EPOCHS, ADD_BS, T=T_G, return_model=True)
    prof["after"][kind] = grad_profile(kind, trained.rec, trained.fc, xg, yg)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for a, when in zip(ax, ("before", "after")):
    for kind in KINDS:
        a.semilogy(np.arange(1, T_G + 1), np.maximum(prof[when][kind], 1e-30), label=NAMES[kind])
    a.set_title(f"{when} training (T = {T_G})"); a.set_xlabel("time step t"); a.legend(fontsize=8)
ax[0].set_ylabel(r"$\|\partial L/\partial h_t\|$")
plt.tight_layout(); plt.savefig("figures/gradient_norms.png", dpi=150); plt.show()

In [ ]:
# [5b: hidden-size sweep {16, 32, 64, 128}, LSTM and GRU, HAR]
for hs_ in (16, 32, 64, 128):
    for seed in SEEDS:
        for kind in ("lstm", "gru"):
            run_logged("har_sweep", kind, "har", seed, hs_)

df = pd.DataFrame(rows); sw = df[df.exp == "har_sweep"]
sweep_table = sw.groupby(["kind", "H"]).agg(test_acc=("test_acc", ms), params=("params", "first"),
                                            sec_per_epoch=("sec_per_epoch", lambda s: f"{s.mean():.2f}"))
print(sweep_table.to_string())
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for kind in ("lstm", "gru"):
    g = sw[sw.kind == kind].groupby("H").test_acc
    ax.errorbar(g.mean().index, g.mean(), yerr=g.std(), marker="o", capsize=3, label=NAMES[kind])
ax.set_xscale("log", base=2); ax.set_xlabel("hidden size"); ax.set_ylabel("test accuracy"); ax.legend()
plt.tight_layout(); plt.savefig("figures/hidden_size_sweep.png", dpi=150); plt.show()

In [ ]:
# [5c: ablation for an ODD student-ID digit: LSTM forget-gate bias 0 vs 1, T = 200]
for fb in (0.0, 1.0):
    for seed in SEEDS:
        run_logged("abl_forget", "lstm", "add", seed, H, T=200, forget_bias=fb)

df = pd.DataFrame(rows); ab = df[df.exp == "abl_forget"].copy()
ab["solved"] = ab.test_mse < 0.05
print(ab.groupby("forget_bias").agg(test_mse=("test_mse", lambda s: f"{s.mean():.4f} ± {s.std():.4f}"),
                                    solved=("solved", lambda s: f"{int(s.sum())}/{len(s)}")).to_string())
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for fb in (0.0, 1.0):
    curves = np.array([json.loads(c) for c in ab[ab.forget_bias == fb].val_curve])
    m, s = curves.mean(0), curves.std(0); ep = np.arange(1, len(m) + 1)
    ax.plot(ep, m, label=f"forget bias = {fb:g}"); ax.fill_between(ep, m - s, m + s, alpha=0.2)
ax.axhline(0.167, color="gray", ls="--"); ax.set_yscale("log")
ax.set_xlabel("epoch"); ax.set_ylabel("validation MSE"); ax.set_title("LSTM, T = 200"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/ablation_forget_bias.png", dpi=150); plt.show()

## Packaging

In [ ]:
# [requirements.txt versions]
import sklearn, matplotlib
with open("requirements.txt", "w") as f:
    for name, mod in (("torch", torch), ("numpy", np), ("pandas", pd), ("matplotlib", matplotlib), ("scikit-learn", sklearn)):
        f.write(f"{name}=={mod.__version__.split('+')[0]}\n")
print(open("requirements.txt").read())

In [ ]:
# [Save tables and bundle outputs for download]
os.makedirs("tables", exist_ok=True)
for name, tab in (("har_table", har_table), ("add_table", add_table), ("sweep_table", sweep_table), ("param_counts", pc_table)):
    tab.to_csv(f"tables/{name}.csv")
with zipfile.ZipFile("problem1_outputs.zip", "w") as z:
    for f in ("results.csv", "results_adding_40ep.csv", "requirements.txt"):
        if os.path.exists(f): z.write(f)
    for folder in ("figures", "tables"):
        for f in sorted(os.listdir(folder)): z.write(os.path.join(folder, f))
print("saved problem1_outputs.zip with:", zipfile.ZipFile("problem1_outputs.zip").namelist())
try:
    from google.colab import files
    files.download("problem1_outputs.zip")
except Exception:
    pass

## 5(d) One-page analysis (write after the runs finish)
Cover, with numbers from the tables above:
1. What the gradient plot says about memory (which model keeps gradient signal back to early steps, and by how many orders of magnitude).
2. Where each model fails as T grows (see the adding-problem table and solved counts).
3. The accuracy / parameters / seconds-per-epoch trade-off.
4. Which model you would put on a battery-powered wearable, with reasons from your tables.

**AI-use statement (for the report):** state the tool (Claude) and what it was used for: data pipeline code, cells, training/experiment code, debugging.